# Locust Load Test
This notebook provides a foundational guide for running load tests against your Databricks Mosaic AI Model Serving endpoints using [Locust](https://locust.io/). Locust is an open-source framework that simulates real-world traffic by spawning a configurable number of concurrent clients or users. It measures key performance indicators—such as latency percentiles and Requests Per Second (RPS)—and captures any errors encountered during testing.

## Terminology
- `Load test` - a type of performance test that evaluates how a system handles a specific amount of traffic or users.
- `Concurrent Requests/Clients` - Client concurrency is the total number of clients that will simultaneously send requests in parallel to a Databricks Model Serving endpoint. In Locust this is called `Locust Users`.
- PXX (P50, P99, etc) - The latency (in milliseconds) for which XX percentile of requests has finished faster than. For example, a P90 of 30ms means that 90% of all requests have finished in 30ms or less.
- `Requests per Second (RPS)` - The number of requests which have completed per second (response is returned from the Databricks Model Serving endpoint to the client).

## How to use this notebook
- Create your endpoint with a "Small" size (concurrency of 4), with scale to zero disabled.
- [Create a Databricks Service Principal](https://docs.databricks.com/aws/en/machine-learning/model-serving/query-route-optimization#fetch-an-oauth-token-and-query-the-endpoint) (SP) and grant the SP "Query" permissions on your model serving endpoint.
- Put the Client ID and Client Secret of your SP into a [Databricks Secret](https://docs.databricks.com/aws/en/security/secrets/example-secret-workflow), or insert them directly in the `Variables` section below.
- Ensure you've properly configured the `input.json` file to contain a functioning payload to your endpoint. This is the payload that will be sent by all concurrent connections to your endpoint. If you're testing an endpoint which is senstive to the size of the payload (for example, a generative LLM) ensure the input payload is representative of how you expect the endpoint to be used. (Make sure you can go to your endpointw webpage and send this payload using the "Use" button and get a valid response!)
- **This notebook will NOT work with Serverless Compute.** Connect to a dedicated compute cluster with `15.4 LTS ML` runtime. To avoid hitting a CPU overload, use a cluster with a lot of cores.
- Make sure you understand the goals of your ML system:
  - P50, P90, P99 latency, Requests per Second.


# Things to know
- Locust relies on CPU resources to run its tests. Depending on payload this facilitates roughly 4000 requests / second per CPU core. The `--processes -1` flag has been set to allow Locust to auto-detect the number of CPU cores on your driver and fully utilize them. 
  - Keep an eye on the locust output, if locust is being bottlenecked by the CPU it will output a message.
- The Model Serving Endpoint concurrency you need to achieve a certain percentile of latency scales linearly with the number of concurrent connections. This means we can test on a small endpoint and calculate the size endpoint you will need in the end before performing a final test.
- In order to load test an even larger scale, you can simply use `fast_load_test.py` while running [Locust in Distributed Setup](https://docs.locust.io/en/2.0.0/running-locust-distributed.html#example).

In [ ]:
# Local prerequisites — run once in your shell, NOT here:
#   uv add locust==2.32.6 gevent==24.11.1 matplotlib pandas python-dotenv
# (or use --with on uv run / pip install if you prefer.)
#
# This cell is a no-op locally; left as a marker for parity with the workspace version.
import sys
print(f'Python: {sys.version.split()[0]}')

In [ ]:
import os
import pandas as pd
import requests
import matplotlib.pyplot as plt
import subprocess
import shlex
import time
from contextlib import contextmanager
from datetime import datetime
import shutil
import pathlib
from math import ceil
from IPython.display import display    # Databricks injects this automatically; locally we import it.

# Variables
### Endpoint
- endpoint_name (Auto-filled) : The name of the serving endpoint.
- endpoint_url (Auto-filled): The route optimized URL of the serving endpoint.
- workspace_url (Auto-filled): the url to your workspace ("https://abc.cloud.databricks.com/")

### Service Principal
- CLIENT_ID : the secret scope location of a databricks service principal client ID
- CLIENT_SECRET : the secret scope location of a databricks service principal client Secret

### Locust
- csv_output_prefix (Auto-filled) : A prefix for the generated resulting CSV files.
- locust_run_time : How many minutes to run the load test? Use suffixes "s" for seconds, "m" for minutes, etc.


In [ ]:
# 1) Anchor cwd at this notebook's folder (load_testing/).
#    VSCode often launches the kernel from the workspace root; this fixes that
#    so subsequent relative paths (fast_load_test.py, input.json, CSVs) all work.
_nb_dir = None
if '__vsc_ipynb_file__' in globals():
    _nb_dir = pathlib.Path(__vsc_ipynb_file__).parent     # VSCode Jupyter ext
elif (pathlib.Path.cwd() / 'fast_load_test.py').exists():
    _nb_dir = pathlib.Path.cwd()                          # Jupyter Lab from load_testing/
else:
    # Fallback: search down from cwd for fast_load_test.py
    for _hit in pathlib.Path.cwd().rglob('fast_load_test.py'):
        _nb_dir = _hit.parent
        break
if _nb_dir is None:
    raise RuntimeError('Could not locate load_testing/ — open the notebook from inside the project, or cd to load_testing/ first.')
if pathlib.Path.cwd().resolve() != _nb_dir.resolve():
    os.chdir(_nb_dir)
print(f'cwd: {pathlib.Path.cwd()}')

# 2) Walk up to find the project .env (CLIENT_ID + CLIENT_SECRET).
_env_path = None
for _candidate in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    _try = _candidate / '.env'
    if _try.exists():
        _env_path = _try
        break
if _env_path is None:
    raise RuntimeError('Could not find .env. Create one at the project root with CLIENT_ID and CLIENT_SECRET.')
for _line in _env_path.read_text().splitlines():
    if '=' in _line and not _line.strip().startswith('#'):
        _k, _, _v = _line.partition('=')
        os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))
print(f'Loaded credentials from {_env_path}')

# 3) Endpoint Configuration
endpoint_name = 'claims-fe-transformer'
endpoint_url = 'https://ea6cbfd5fac74702ad2b8fb5f2f45596.serving.cloud.databricks.com/1444828305810485/serving-endpoints/claims-fe-transformer/invocations'
# Make sure to add '/' at the end of the path
workspace_url = 'https://e2-demo-field-eng.cloud.databricks.com/'

# Service Principal (loaded from .env above)
CLIENT_ID = os.environ['CLIENT_ID']
CLIENT_SECRET = os.environ['CLIENT_SECRET']

# Locust Variables (save as strings!)
locust_run_time = '5m'  # In minutes
csv_output_prefix = 'load_test'
endpoint_prefix = endpoint_url.split('serving-endpoints')[0]
endpoint_id = endpoint_prefix.split('.')[0].split('//')[1]

print(f'endpoint_name : {endpoint_name}')
print(f'endpoint_id   : {endpoint_id}')
print(f'CLIENT_ID     : {CLIENT_ID[:8]}... (len={len(CLIENT_ID)})')

In [0]:
os.environ["DATABRICKS_WORKSPACE_URL"] = workspace_url
os.environ["ENDPOINT_ID"] = endpoint_id
os.environ["CLIENT_ID"] = CLIENT_ID
os.environ["CLIENT_SECRET"] = CLIENT_SECRET
os.environ["DATABRICKS_ENDPOINT_NAME"] = endpoint_name

Define helper functions

In [0]:
def check_for_CPU_overload(line : str) -> None:
  """Watches for CPU overload messages in the locust output and raises an exception if found."""
  if "CPU" in line:
      raise Exception("CPU Overloaded, restart this notebook with a single node cluster with more available cores.")

def run_locust_test(host : str, users : int, spawn_rate : int, run_time : str, csv_output_prefix : str, locust_file : str = "fast_load_test.py", verbose : bool = False):
  """Spawns a locust processs with the specified parameters."""
  locust_command = f"""locust --host={host} --users={users} --spawn-rate={spawn_rate} --run-time={run_time} --headless --locustfile={locust_file} --csv={csv_output_prefix} --processes -1 --stop-timeout 10"""

  process = subprocess.Popen(shlex.split(locust_command), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
  # While the process is running, watch for CPU overload
  while True:
      line = process.stdout.readline()
      if not line and process.poll() is not None:
          break
      if line:
        check_for_CPU_overload(line)
        if verbose:
          print(line.strip())

# Step 1
First we're going to run a really short test (30 seconds) just to warm the endpoint up!

In [0]:
file_path = str(pathlib.Path("fast_load_test.py").resolve())
file_path  = f'"{file_path}"' # Put it into quotes, in case there is a space in the folder path.
run_locust_test(endpoint_prefix, 4, 1, "30s", f"{csv_output_prefix}_warmup", locust_file=file_path, verbose=True)

[2026-05-05 06:08:49,992] 0402-052914-ad586jgy-10-0-18-207/INFO/locust.main: Starting Locust 2.32.6
[2026-05-05 06:08:50,181] 0402-052914-ad586jgy-10-0-18-207/INFO/locust.runners: 0402-052914-ad586jgy-10-0-18-207_72dc61a02a464cf7aba0c819dbf84f4a (index 0) reported as ready. 1 workers connected.
[2026-05-05 06:08:50,182] 0402-052914-ad586jgy-10-0-18-207/INFO/locust.runners: 0402-052914-ad586jgy-10-0-18-207_060cea5ce25d4731832f5842291cfb50 (index 1) reported as ready. 2 workers connected.
[2026-05-05 06:08:50,182] 0402-052914-ad586jgy-10-0-18-207/INFO/locust.runners: 0402-052914-ad586jgy-10-0-18-207_fbe06ebc465949689a9cc20cbca8c2bb (index 2) reported as ready. 3 workers connected.
[2026-05-05 06:08:50,182] 0402-052914-ad586jgy-10-0-18-207/INFO/locust.runners: 0402-052914-ad586jgy-10-0-18-207_1fd026af659f4c0d8f4abb354bc5bd43 (index 3) reported as ready. 4 workers connected.
[2026-05-05 06:08:50,182] 0402-052914-ad586jgy-10-0-18-207/INFO/locust.runners: 0402-052914-ad586jgy-10-0-18-207_8f1

# Step 2
Next we are going to test out different number of client connections on our Small endpoint to determine the acceptable latency.

In [0]:
client_connections_test_values = [2,3,4,5] # Number of client connections to test
for client_connections in client_connections_test_values:
  print(f"============== {client_connections} Client Side connections test ==============")
  run_locust_test(
    endpoint_prefix, 
    client_connections, 
    4, 
    locust_run_time, 
    f"{csv_output_prefix}_{client_connections}", 
    locust_file=file_path, 
    verbose=True
  )

============== 2 Client Side connections test ==============
[2026-05-05 06:10:03,975] 0402-052914-ad586jgy-10-0-18-207/INFO/locust.main: Starting Locust 2.32.6
[2026-05-05 06:10:04,145] 0402-052914-ad586jgy-10-0-18-207/INFO/locust.runners: 0402-052914-ad586jgy-10-0-18-207_03c5a78821e044e3914652e55ac3b26b (index 0) reported as ready. 1 workers connected.
[2026-05-05 06:10:04,145] 0402-052914-ad586jgy-10-0-18-207/INFO/locust.runners: 0402-052914-ad586jgy-10-0-18-207_1c531ec3250942b7bd7f47ce89a52447 (index 1) reported as ready. 2 workers connected.
[2026-05-05 06:10:04,145] 0402-052914-ad586jgy-10-0-18-207/INFO/locust.runners: 0402-052914-ad586jgy-10-0-18-207_5ddd011ff14d4df8b6068fb2c4ad2126 (index 2) reported as ready. 3 workers connected.
[2026-05-05 06:10:04,145] 0402-052914-ad586jgy-10-0-18-207/INFO/locust.runners: 0402-052914-ad586jgy-10-0-18-207_c27ae65e9ec14e609a90d0a78241146b (index 3) reported as ready. 4 workers connected.
[2026-05-05 06:10:04,146] 0402-052914-ad586jgy-10-0-18-

Print any observed failures.

In [0]:
all_failures_csv = [pd.read_csv(f"{csv_output_prefix}_{i}_failures.csv") for i in client_connections_test_values]
failures = [csv["Occurrences"].iloc[0] if not csv.empty else 0 for csv in all_failures_csv]
failure_df = pd.DataFrame({
  "Client Connections": client_connections_test_values,
  "Failures": failures
})
display(failure_df)

---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
File <command-6976838247564028>, line 1
----> 1 all_failures_csv = [pd.read_csv(f"{csv_output_prefix}_{i}_failures.csv") for i in client_connections_test_values]
      2 failures = [csv["Occurrences"].iloc[0] if not csv.empty else 0 for csv in all_failures_csv]
      3 failure_df = pd.DataFrame({
      4   "Client Connections": client_connections_test_values,
      5   "Failures": failures
      6 })

File /databricks/python/lib/python3.12/site-packages/pandas/util/_decorators.py:211, in deprecate_kwarg.<locals>._deprecate_kwarg.<locals>.wrapper(*args, **kwargs)
    209     else:
    210         kwargs[new_arg_name] = new_arg_value
--> 211 return func(*args, **kwargs)

File /databricks/python/lib/python3.12/site-packages/pandas/util/_decorators.py:331, in deprecate_nonkeyword_arguments.<locals>.decorate.<locals>.wrapper(*args, **kwargs)

Print any received exceptions.

In [0]:
all_exceptions_csv = [pd.read_csv(f"{csv_output_prefix}_{i}_exceptions.csv") for i in client_connections_test_values]
exceptions = [csv["Count"].iloc[0] if not csv.empty else 0 for csv in all_exceptions_csv]
exceptions_df = pd.DataFrame({
  "Client Connections": client_connections_test_values,
  "Exceptions": exceptions
})
display(exceptions_df)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

Read summary CSV's for all load test runs, and combine into a readable table.

In [0]:
combined_latency_df_list = []

percentiles_to_keep = ["50%", "80%", "90%", "95%", "99%"]
for i in client_connections_test_values:
  current_latency_df = pd.read_csv(f"{csv_output_prefix}_{i}_stats_history.csv")
  current_latency_df = current_latency_df[current_latency_df["User Count"] == i]
  current_latency_df = current_latency_df[["User Count", "Requests/s", "Total Average Response Time", "Total Min Response Time"] + percentiles_to_keep]
  current_latency_df = current_latency_df.groupby("User Count")[["Requests/s", "Total Average Response Time", "Total Min Response Time"] + percentiles_to_keep].mean().reset_index()

  combined_latency_df_list.append(current_latency_df)

# Concatenate all DataFrames in the list
combined_latency_results_df = pd.concat(combined_latency_df_list, ignore_index=True)
display(combined_latency_results_df)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:434)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:477)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:510)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:616)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:643)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:49)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:293)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:62)
	at com.databricks.logging.AttributionContext$.withValue(Attr

Plot the observed Latency percentiles (y-axis) for different Client Concurrency values (x-axis).

In [0]:
# Plot the data
plt.figure(figsize=(10, 6))
for percentile in percentiles_to_keep:
  plt.plot(combined_latency_results_df["User Count"], combined_latency_results_df[percentile], label=percentile)

# Add labels and title
plt.xlabel("Concurrent Users")
plt.ylabel("Latency (ms)")
plt.title("Latency vs Concurrent Users")
plt.legend()

# Improve tick marks readability
plt.xticks(rotation=45)
plt.grid(True)

# Display the plot
plt.show()

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:434)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:477)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:510)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:616)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:643)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:49)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:293)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:62)
	at com.databricks.logging.AttributionContext$.withValue(Attr

# Step 3
Consider the below latency table and select (by configuring the `USER_COUNT_SELECTION` parameter below) the row which provides the maxmimum acceptable latencies for your application. Consider also whether the failure/exception rate for that selection is okay for your application (in the `failures_df` and `exceptions_df` dataframes).

In [0]:
latency_table = combined_latency_results_df[["User Count"] + percentiles_to_keep]
display(latency_table)

# Step 4
Enter your selection below and your applications desired *Requests Per Second* (RPS)

In [ ]:
# Pick the user count from the latency table above. 4 is a reasonable default for the Small endpoint.
USER_COUNT_SELECTION = 4    # value between 2 - 10
REQUESTS_PER_SECOND = 50    # your application's target RPS

In [0]:
client_to_worker_ratio = USER_COUNT_SELECTION / 4 # Assumes user is testing this on a small endpoint with 4 concurrency
rps = combined_latency_results_df[combined_latency_results_df["User Count"] == USER_COUNT_SELECTION]["Requests/s"].iloc[0]
rps_multiple = ceil(REQUESTS_PER_SECOND / rps)

# Step 5
Resize your endpoint to have the concurrency given below and rerun the load test!

In [0]:
endpoint_concurrency_needed = 4 * rps_multiple
client_connections_needed = USER_COUNT_SELECTION * rps_multiple

print(f"Please now resize your endpoint to have a concurrency of {endpoint_concurrency_needed}, then run the load test with {client_connections_needed} locust users.")

In [0]:
# Warmup endpoint since it has restarted with new configuration
run_locust_test(endpoint_prefix, client_connections_needed, client_connections_needed, "30s", f"{csv_output_prefix}_warmup", locust_file=file_path, verbose=True)

In [0]:
run_locust_test(
  endpoint_prefix,
  client_connections_needed,
  10,
  "60m",
  f"{csv_output_prefix}_final",
  locust_file=file_path,
  verbose=True
)

# Step 6
View load test results!

In [0]:
# Print any failures!
all_failures_csv = pd.read_csv(f"{csv_output_prefix}_final_failures.csv")
all_failures_csv

In [0]:
final_load_test_result = pd.read_csv(f"{csv_output_prefix}_final_stats.csv")
final_load_test_result